Este código corresponde aos testes realizados, para validar a abordagem de polinômios de quarta ordem com window_size_max = 21 e que **armazena** valores que não satisfazem as condições de ativação dos sensores enquanto utiliza os strikes para continuar buscando uma ativação próxima. Para esta avaliação foram utilizados os conjuntos de dados exec1 e exec2. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from matplotlib.dates import DateFormatter
from collections import Counter

In [227]:
path = 'exps/exp4-22-07/ev8.csv'

# Abordagem completa

## Pré-tratamento inicial

In [228]:
#Carregando os dados
data = pd.read_csv(path)

In [229]:
# Estruturando coluna de tempo
data = data.sort_values('id')
data["measured_at"] = data["measured_at"].astype(str)
data["minute"] = data["measured_at"].str[14:16].astype(int)  # Extrai os minutos
data["second"] = data["measured_at"].str[17:19].astype(int)  # Extrai os segundos
data["millisecond"] = data["measured_at"].str[20:].fillna("0").replace("", "0").astype(int)
data['measured_at'] = (data["minute"].astype(int) * 60 * 1000) + (data["second"].astype(int) * 1000) + data["millisecond"].astype(int)
base_datetime = pd.to_datetime(data['epoch_start'], unit='ns').dt.floor('H')


# --- PASSO 2: Criar o deslocamento de tempo (Timedelta) ---
# Vamos converter as colunas de tempo em um objeto de duração (Timedelta).
time_offset = (pd.to_timedelta(data['minute'], unit='m') +
               pd.to_timedelta(data['second'], unit='s') +
               pd.to_timedelta(data['millisecond'], unit='ms'))


# --- PASSO 3: Criar a coluna final somando a base e o deslocamento ---
# O resultado é um datetime preciso construído com as colunas que você pediu.
data['datetime_construido'] = base_datetime + time_offset

# Agrega medidas
data['soma_distancias_cm'] = data['distance_cm_inside'] + data['distance_cm_outside']

/tmp/ipykernel_27267/457586069.py:8: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  base_datetime = pd.to_datetime(data['epoch_start'], unit='ns').dt.floor('H')


## Funções

In [230]:
#Função dividir o dataframe em eventos de interesse

def split_events_subdfs_with_head_tail(
    df: pd.DataFrame,
    inside_col: str = "distance_cm_inside",
    outside_col: str = "distance_cm_outside",
    sum_col: str | None = "soma_distancias_cm",
    min_len: int = 3,
    strikes_to_split: int = 3,
    head_len: int = 3,
    tail_len: int = 3,   # normalmente igual a strikes_to_split
):
    df = df.copy()

    inside = df[inside_col]
    s = df[sum_col] if (sum_col is not None and sum_col in df.columns) else (inside + df[outside_col])

    #Ta diferente do processo de determinar estados de inatividade
    ok = ~((s < 500) & (s >= 270)) & ~((inside < 9) & (inside >= 3))

    idxs = list(df.index)
    subdfs = []

    in_event = False
    body = []
    tail = []
    strikes = 0
    start_pos = 0

    for pos, idx in enumerate(idxs):
        if ok.loc[idx]:
            if not in_event:
                in_event = True
                body = []
                tail = []
                strikes = 0
                start_pos = pos  # onde começa o body (pra pegar o head depois)
            else:
                if tail:
                    body.extend(tail)

            body.append(idx)
            tail = []
            strikes = 0

        else:
            if not in_event:
                continue

            strikes += 1
            tail.append(idx)
            if len(tail) > tail_len:
                tail = tail[-tail_len:]

            if strikes >= strikes_to_split:
                # fecha evento: head + body + tail
                if len(body) >= min_len:
                    head_start = max(0, start_pos - head_len)
                    head = idxs[head_start:start_pos]
                    event_idxs = body + tail
                    subdfs.append(df.loc[event_idxs].copy())

                in_event = False
                body = []
                tail = []
                strikes = 0

    # se terminou no fim do df com evento aberto, fecha sem tail (não teve strikes suficientes)
    if in_event and len(body) >= min_len:
        head_start = max(0, start_pos - head_len)
        head = idxs[head_start:start_pos]
        event_idxs = body + tail
        subdfs.append(df.loc[event_idxs].copy())

    return subdfs

In [ ]:
#Função para calcular threshold
def calculate_threshold(df):
    """
    Calcula um limite (threshold) com base nas distâncias dentro e fora.

    Args:
        df (pd.DataFrame): O DataFrame de entrada contendo as colunas 'distance_cm_inside' e 'distance_cm_outside'.

    Returns:
        float: O valor do limite calculado.
    """


    mean_outside = df['outside_smooth'].mean()
    mean_inside = df['inside_smooth'].mean()
    mean_all = (mean_outside + mean_inside)/2

    if mean_outside >= mean_inside:
        threshold = mean_all
    else:
        threshold = mean_all
    
    return threshold

In [232]:
# Função para determinar estados do sensor
def determine_state_of_sensor(row):
    if row['inside_smooth'] < threshold and row['outside_smooth'] < threshold:
        #ambos sensoes ativados
        return 3
    elif row['inside_smooth'] < threshold and row['outside_smooth'] > threshold:
        #sensor interno ativado
        return 1
    elif row['inside_smooth'] > threshold and row['outside_smooth'] < threshold:
        #sensor externo ativado
        return 2
    else:
        return 0

In [233]:
# Função para detectar entrada ou saida
def detectar_entrada_saida(seq, entrou=(2, 3, 1), saiu=(1, 3, 2), verbose=True):
    resultados = []
    entrou = list(entrou)
    saiu = list(saiu)

    for i in range(len(seq) - 2):
        janela = seq[i:i+3]

        if janela == entrou:
            evento = {"tipo": "entrou", "ini": i, "fim": i+2, "janela": janela}
            resultados.append(evento)
            if verbose:
                print(f"alguém entrou (pos {i}-{i+2})")

        elif janela == saiu:
            evento = {"tipo": "saiu", "ini": i, "fim": i+2, "janela": janela}
            resultados.append(evento)
            if verbose:
                print(f"alguém saiu (pos {i}-{i+2})")

    # ✅ caso nenhum evento seja detectado
    if not resultados:
        falso_evento = {"tipo": "falso_evento"}
        resultados.append(falso_evento)
        if verbose:
            print("falso evento")

    return resultados


## Criando subdataframes

In [234]:
eventos = split_events_subdfs_with_head_tail(
    df=data,
    min_len=3,
    strikes_to_split=5,
    head_len=3,
    tail_len=3,
)

print(len(eventos))
for i, evento in enumerate(eventos):
    print(f"Evento {i}: tamanho = {len(evento)}")

3
Evento 0: tamanho = 11
Evento 1: tamanho = 23
Evento 2: tamanho = 25


## Iterando sobre os dataframes

In [235]:
# Sequência de etapas para gerar resultados
for i, evento in enumerate(eventos):
    print(f"Evento {path}, subdataframe {i}")
    print(f"Tamanho do evento: {len(evento)}")
    window_size = len(evento) if len(evento) <= 21 else 21
    window_size = window_size if window_size % 2 == 1 else window_size - 1
    poly_order = 4
    poly_order = poly_order if window_size > poly_order else (window_size - 1)
    df = evento
    df['inside_smooth'] = savgol_filter(df['distance_cm_inside'], window_size, poly_order, mode='nearest')
    df['outside_smooth'] = savgol_filter(df['distance_cm_outside'], window_size, poly_order, mode='nearest')
    threshold = calculate_threshold(df)
    df['state'] = df.apply(determine_state_of_sensor, axis=1)
    print(df["state"].to_list())
    janela = df["state"].loc[df["state"].ne(df["state"].shift())].to_list()
    results = detectar_entrada_saida(janela)
    print(janela)

Evento exps/exp4-22-07/ev8.csv, subdataframe 0
Tamanho do evento: 11
outside---19.313156573825662---178.15087942361748---186.2773892774006
inside---72.61896762270301---81.4625980080575---69.30303030303449
[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]
falso evento
[1, 0]
Evento exps/exp4-22-07/ev8.csv, subdataframe 1
Tamanho do evento: 23
outside---47.66638781992722---118.41727592633598---122.70926677301598
inside---53.23006001702947---98.7785515718135---120.99231198194713
[3, 0, 2, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 3, 3, 3, 3, 2, 2, 2, 0, 0, 0]
alguém saiu (pos 4-6)
[3, 0, 2, 0, 1, 3, 2, 0]
Evento exps/exp4-22-07/ev8.csv, subdataframe 2
Tamanho do evento: 25
outside---27.588432993578394---133.52639009281975---130.85377382068265
inside---33.401056952490435---37.54525915810873---38.69404457433431
[0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
falso evento
[0, 1]
